# 1. Data Understanding & Setup

In [19]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

In [20]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [32]:
df = pd.read_csv('data_amazon.xlsx')


In [22]:
# 1-2: Negative (0), 3: Neutral (1), 4-5: Positive (2)
def label_sentiment(rating):
    if rating <= 2: return 0
    if rating == 3: return 1
    return 2

In [23]:
df['Sentiment'] = df['Cons_rating'].apply(label_sentiment)

print(f"Dataset Shape: {df.shape}")
print("Class Distribution:\n", df['Sentiment'].value_counts())

Dataset Shape: (49338, 10)
Class Distribution:
 Sentiment
2    36840
0     7148
1     5350
Name: count, dtype: int64


# 2. NLP Preprocessing

In [24]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [25]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [26]:
df['Cleaned_Review'] = df['Review'].apply(preprocess_text)
print("Sample Preprocessed Text:\n", df[['Review', 'Cleaned_Review']].head())

Sample Preprocessed Text:
                                               Review  \
0  Absolutely wonderful - silky and sexy and comf...   
1  Love this dress!  it's sooo pretty.  i happene...   
2  I had such high hopes for this dress and reall...   
3  I love, love, love this jumpsuit. it's fun, fl...   
4  This shirt is very flattering to all due to th...   

                                      Cleaned_Review  
0        absolutely wonderful silky sexy comfortable  
1  love dress sooo pretty happened find store im ...  
2  high hope dress really wanted work initially o...  
3  love love love jumpsuit fun flirty fabulous ev...  
4  shirt flattering due adjustable front tie perf...  


# 3.Feature Engineering

In [27]:
X = df['Cleaned_Review']
y = df['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
cv = CountVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

In [29]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Model Building

In [30]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced')
}

results = []

# 5. Model Evaluation

In [31]:
def evaluate_models(X_train_set, X_test_set, feature_name):
    print(f"\n--- Performance using {feature_name} ---")
    for name, model in models.items():
        model.fit(X_train_set, y_train)
        y_pred = model.predict(X_test_set)

        acc = accuracy_score(y_test, y_pred)
        results.append({"Model": name, "Features": feature_name, "Accuracy": acc})

        print(f"Model: {name}")
        print(classification_report(y_test, y_pred))

# BoW
evaluate_models(X_train_bow, X_test_bow, "Bag of Words")

# TF-IDF
evaluate_models(X_train_tfidf, X_test_tfidf, "TF-IDF")


--- Performance using Bag of Words ---
Model: Logistic Regression
              precision    recall  f1-score   support

           0       0.47      0.64      0.54      1373
           1       0.29      0.43      0.35      1060
           2       0.94      0.81      0.87      7435

    accuracy                           0.75      9868
   macro avg       0.57      0.63      0.59      9868
weighted avg       0.80      0.75      0.77      9868

Model: Naive Bayes
              precision    recall  f1-score   support

           0       0.59      0.61      0.60      1373
           1       0.36      0.41      0.38      1060
           2       0.91      0.89      0.90      7435

    accuracy                           0.80      9868
   macro avg       0.62      0.63      0.63      9868
weighted avg       0.81      0.80      0.80      9868

Model: Decision Tree
              precision    recall  f1-score   support

           0       0.42      0.50      0.45      1373
           1       0.2

# 6. Comparisons & Insights

# NLP Evaluation: Performance Comparison & Insights

This report summarizes the performance of three classification models (Logistic Regression, Naive Bayes, and Decision Tree) across two text vectorization techniques (**Bag of Words** and **TF-IDF**).

---

## 📊 Performance Summary Table

| Vectorizer | Model | Accuracy | Macro F1-Score | Weighted F1-Score |
| :--- | :--- | :--- | :--- | :--- |
| **BoW** | Logistic Regression | 0.75 | 0.59 | 0.77 |
| **BoW** | **Naive Bayes (Best Balance)** | **0.80** | **0.63** | **0.80** |
| **BoW** | Decision Tree | 0.71 | 0.52 | 0.72 |
| **TF-IDF** | Logistic Regression | 0.77 | 0.63 | 0.79 |
| **TF-IDF** | Naive Bayes | 0.81 | 0.52 | 0.76 |
| **TF-IDF** | Decision Tree | 0.70 | 0.50 | 0.72 |

---

## Key Insights

### 1. The Vectorizer Battle: BoW vs. TF-IDF
* **Logistic Regression loves TF-IDF:** It saw a significant boost in Accuracy ($0.75 \rightarrow 0.77$) and Macro F1 ($0.59 \rightarrow 0.63$). By weighting unique words higher, the model stopped getting distracted by common "stop words" and identified minority classes better.
* **Naive Bayes & the TF-IDF Trap:** While Naive Bayes achieved its highest Accuracy ($0.81$) with TF-IDF, this is **misleading**. The Recall for Class 1 crashed to **0.06** (meaning it missed 94% of Class 1 cases!). **BoW is the safer choice** here as it maintains a much better Macro F1 ($0.63$).

### 2. The "Best" Performer
* **Overall MVP:** **Naive Bayes with Bag of Words**. Even though it's a simple probabilistic model, it provided the most reliable results across all three classes.
* **Precision Specialist:** **TF-IDF + Naive Bayes** has the highest Precision for Class 0 ($0.74$), but it achieves this by being overly "cautious," leading to very low Recall.
